# NMT Assignment
Sanskrit to English Neural Machine Translation

1. Title & Student Information

# Assignment 2: Sanskrit-to-English Neural Machine Translation

Name: Niyanta Prasad
Roll No: G25AIT1105

## Preprocessing, Vocabulary, Dataset, Encoder, Attention, Decoder, Training, Evaluation, Submission Generation
2. Dependency Installation

In [ ]:
!pip install sacrebleu bert-score sentencepiece nltk tqdm

3. Imports

In [ ]:
import os
import re
import time
import random
import numpy as np
import pandas as pd

from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from tqdm import tqdm

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(device)

4. Load Dataset

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
train_sa = pd.read_csv("train_sa_10000.csv")
train_en = pd.read_csv("train_en_10000.csv")

dev_sa = pd.read_csv("dev_sa_1000.csv")
dev_en = pd.read_csv("dev_en_1000.csv")

test_sa = pd.read_csv("test_sa_1000.csv")

train_df = pd.DataFrame({
    "sa": train_sa["Sentence_sa"],
    "en": train_en["Sentence_en"]
})

dev_df = pd.DataFrame({
    "sa": dev_sa["Sentence_sa"],
    "en": dev_en["Sentence_en"]
})

print(train_df.shape)
print(dev_df.shape)

5. Clean Text

In [ ]:
def clean_text(text):

    text = str(text).lower()

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()

train_df["sa"] = train_df["sa"].apply(clean_text)
train_df["en"] = train_df["en"].apply(clean_text)

dev_df["sa"] = dev_df["sa"].apply(clean_text)
dev_df["en"] = dev_df["en"].apply(clean_text)

6. Vocabulary

In [ ]:
PAD = "<pad>"
SOS = "<sos>"
EOS = "<eos>"
UNK = "<unk>"

def build_vocab(texts,min_freq=1):

    counter = Counter()

    for line in texts:
        counter.update(line.split())

    vocab = [
        PAD,
        SOS,
        EOS,
        UNK
    ]

    for token,count in counter.items():

        if count >= min_freq:
            vocab.append(token)

    stoi = {
        word:i
        for i,word in enumerate(vocab)
    }

    itos = {
        i:word
        for word,i in stoi.items()
    }

    return stoi,itos

sa_stoi, sa_itos = build_vocab(
    train_df["sa"]
)

en_stoi, en_itos = build_vocab(
    train_df["en"]
)

SRC_VOCAB = len(sa_stoi)
TGT_VOCAB = len(en_stoi)

print(SRC_VOCAB)
print(TGT_VOCAB)

7. Dataset

In [ ]:
def encode(text,vocab):

    return [
        vocab.get(word,vocab[UNK])
        for word in text.split()
    ]

class NMTDataset(Dataset):

    def __init__(self,df):

        self.df=df

    def __len__(self):

        return len(self.df)

    def __getitem__(self,idx):

        src = self.df.iloc[idx]["sa"]
        tgt = self.df.iloc[idx]["en"]

        src_ids = (
            [sa_stoi[SOS]]
            + encode(src,sa_stoi)
            + [sa_stoi[EOS]]
        )

        tgt_ids = (
            [en_stoi[SOS]]
            + encode(tgt,en_stoi)
            + [en_stoi[EOS]]
        )

        return (
            torch.tensor(src_ids),
            torch.tensor(tgt_ids)
        )

8. Collate Function

In [ ]:
def collate_fn(batch):

    src,tgt = zip(*batch)

    src = nn.utils.rnn.pad_sequence(
        src,
        batch_first=True,
        padding_value=sa_stoi[PAD]
    )

    tgt = nn.utils.rnn.pad_sequence(
        tgt,
        batch_first=True,
        padding_value=en_stoi[PAD]
    )

    return src,tgt

9. Dataloaders

In [ ]:
train_loader = DataLoader(
    NMTDataset(train_df),
    batch_size=64,
    shuffle=True,
    collate_fn=collate_fn
)

dev_loader = DataLoader(
    NMTDataset(dev_df),
    batch_size=64,
    shuffle=False,
    collate_fn=collate_fn
)

10. Encoder

In [ ]:
class Encoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        emb_dim=256,
        hidden_dim=256
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            emb_dim,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            emb_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )

    def forward(self,x):

        emb = self.embedding(x)

        outputs,(hidden,cell) = self.lstm(
            emb
        )

        return outputs,hidden,cell

11. Attention

In [ ]:
class Attention(nn.Module):

    def __init__(self, hidden_size):
        super().__init__()

        # hidden state from decoder: 512
        # encoder outputs: 512
        # concatenated = 1024

        self.attn = nn.Linear(
            hidden_size * 2,
            hidden_size
        )

        self.v = nn.Linear(
            hidden_size,
            1,
            bias=False
        )

    def forward(
        self,
        hidden,
        encoder_outputs
    ):
        """
        hidden:
            (batch_size, 512)

        encoder_outputs:
            (batch_size, seq_len, 512)
        """

        batch_size = encoder_outputs.shape[0]
        seq_len = encoder_outputs.shape[1]

        hidden = hidden.unsqueeze(1)

        hidden = hidden.repeat(
            1,
            seq_len,
            1
        )

        combined = torch.cat(
            (
                hidden,
                encoder_outputs
            ),
            dim=2
        )

        # combined shape:
        # (batch_size, seq_len, 1024)

        energy = torch.tanh(
            self.attn(combined)
        )

        # energy shape:
        # (batch_size, seq_len, 512)

        attention = self.v(
            energy
        ).squeeze(2)

        # attention shape:
        # (batch_size, seq_len)

        return torch.softmax(
            attention,
            dim=1
        )
    
    src, tgt = next(iter(train_loader))

src = src.to(device)
tgt = tgt.to(device)

output = model(src, tgt)

print(output.shape)

12. Decoder

In [ ]:
class Decoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        emb_dim=256,
        hidden=512
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            emb_dim
        )

        self.attention = Attention(
            hidden
        )

        self.lstm = nn.LSTM(
            emb_dim+hidden,
            hidden,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden,
            vocab_size
        )

    def forward(
        self,
        token,
        hidden,
        cell,
        encoder_outputs
    ):

        token = token.unsqueeze(1)

        emb = self.embedding(token)

        weights = self.attention(
            hidden[-1],
            encoder_outputs
        )

        context = torch.bmm(
            weights.unsqueeze(1),
            encoder_outputs
        )

        lstm_input = torch.cat(
            [emb,context],
            dim=2
        )

        output,(hidden,cell)=self.lstm(
            lstm_input,
            (hidden,cell)
        )

        prediction = self.fc(
            output.squeeze(1)
        )

        return prediction,hidden,cell

13. Seq2Seq

In [ ]:
class Seq2Seq(nn.Module):

    def __init__(
        self,
        encoder,
        decoder
    ):
        super().__init__()

        self.encoder=encoder
        self.decoder=decoder

    def forward(
        self,
        src,
        tgt,
        teacher_ratio=0.5
    ):

        batch=tgt.size(0)
        tgt_len=tgt.size(1)

        outputs=torch.zeros(
            batch,
            tgt_len,
            TGT_VOCAB
        ).to(device)

        encoder_outputs,h,c = self.encoder(src)

        h = torch.cat(
            (h[0:1],h[1:2]),
            dim=2
        )

        c = torch.cat(
            (c[0:1],c[1:2]),
            dim=2
        )

        inp = tgt[:,0]

        for t in range(1,tgt_len):

            pred,h,c = self.decoder(
                inp,
                h,
                c,
                encoder_outputs
            )

            outputs[:,t]=pred

            top1=pred.argmax(1)

            inp = (
                tgt[:,t]
                if random.random()
                < teacher_ratio
                else top1
            )

        return outputs

14. Create Model

In [ ]:
encoder = Encoder(
    SRC_VOCAB
)

decoder = Decoder(
    TGT_VOCAB
)

model = Seq2Seq(
    encoder,
    decoder
).to(device)

criterion = nn.CrossEntropyLoss(
    ignore_index=en_stoi[PAD]
)

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

15. Train

In [ ]:
EPOCHS = 15

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for src,tgt in tqdm(train_loader):

        src=src.to(device)
        tgt=tgt.to(device)

        optimizer.zero_grad()

        output=model(src,tgt)

        output = output[:,1:].reshape(
            -1,
            TGT_VOCAB
        )

        tgt2 = tgt[:,1:].reshape(-1)

        loss = criterion(
            output,
            tgt2
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        epoch+1,
        total_loss/len(train_loader)
    )

16. Save Model

In [ ]:
torch.save(
    model.state_dict(),
    "best_model.pt"
)

17. Greedy Translation

In [ ]:
def translate(sentence,max_len=40):

    model.eval()

    tokens = (
        [sa_stoi[SOS]]
        + encode(
            clean_text(sentence),
            sa_stoi
        )
        + [sa_stoi[EOS]]
    )

    src = torch.tensor(
        tokens
    ).unsqueeze(0).to(device)

    with torch.no_grad():

        encoder_outputs,h,c = model.encoder(src)

        h = torch.cat(
            (h[0:1],h[1:2]),
            dim=2
        )

        c = torch.cat(
            (c[0:1],c[1:2]),
            dim=2
        )

    token = torch.tensor(
        [en_stoi[SOS]]
    ).to(device)

    result=[]

    for _ in range(max_len):

        pred,h,c = model.decoder(
            token,
            h,
            c,
            encoder_outputs
        )

        token = pred.argmax(1)

        word = en_itos[
            token.item()
        ]

        if word==EOS:
            break

        result.append(word)

    return " ".join(result)

18. BLEU

In [ ]:
from sacrebleu import corpus_bleu

preds=[]
refs=[]

for i in range(len(dev_df)):

    preds.append(
        translate(
            dev_df.iloc[i]["sa"]
        )
    )

    refs.append(
        dev_df.iloc[i]["en"]
    )

bleu = corpus_bleu(
    preds,
    [refs]
)

print(
    "BLEU:",
    bleu.score
)

19. BERTScore

In [ ]:
from bert_score import score

P,R,F1 = score(
    preds,
    refs,
    lang="en"
)

print(
    F1.mean().item()
)

20. Submission File

In [ ]:
translations=[]

for s in test_sa["Sentence_sa"]:

    translations.append(
        translate(s)
    )

submission = pd.DataFrame({

    "Source_id":
        test_sa["Source_id"],

    "Translation":
        translations
})

submission.to_csv(
    "submission.csv",
    index=False
)

submission.head()

21. Download

In [ ]:
from google.colab import files

files.download(
    "submission.csv"
)